In [5]:
import re
import tldextract
import math
from urllib.parse import urlparse

def shannon_entropy(string):
    prob = [float(string.count(c)) / len(string) for c in dict.fromkeys(list(string))]
    entropy = -sum([p * math.log2(p) for p in prob])
    return entropy

def has_ip(domain):
    pattern = r'(\d{1,3}\.){3}\d{1,3}'
    return 1 if re.search(pattern, domain) else 0

def count_digits(s):
    return sum(c.isdigit() for c in s)

def count_letters(s):
    return sum(c.isalpha() for c in s)

def count_special_chars(s):
    return len(re.findall(r'[^a-zA-Z0-9]', s))

def count_words(s):
    words = re.split(r'[\W_]+', s)
    words = [w for w in words if w]
    return len(words)

suspicious_keywords = [
    'login', 'secure', 'update', 'bank', 'account',
    'verify', 'paypal', 'signin', 'confirm', 'free',
    'webscr', 'ebay', 'amazon'
]

def keyword_count(url):
    return sum(word in url.lower() for word in suspicious_keywords)

def extract_features(url):

    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    domain = ext.domain
    subdomain = ext.subdomain
    path = parsed.path
    
    features = {}
    
    # Basic Length Features
    features['url_length'] = len(url)
    features['domain_length'] = len(domain)
    features['subdomain_length'] = len(subdomain)
    features['path_length'] = len(path)
    
    # Count Features
    features['._count'] = url.count('.')
    features['-_count'] = url.count('-')
    features['__count'] = url.count('_')
    features['/_count'] = url.count('/')
    features['?_count'] = url.count('?')
    features['=_count'] = url.count('=')
    features['@_count'] = url.count('@')
    
    # Digit / Letter Features
    features['digit_count'] = count_digits(url)
    features['letter_count'] = count_letters(url)
    features['special_char_count'] = count_special_chars(url)
    
    # Ratio Features
    features['digit_ratio'] = features['digit_count'] / len(url)
    features['letter_ratio'] = features['letter_count'] / len(url)
    
    # Domain-based
    features['has_ip'] = has_ip(url)
    features['entropy'] = shannon_entropy(url)
    
    # Word-based
    features['word_count'] = count_words(url)
    features['keyword_count'] = keyword_count(url)
    
    # TLD suspicious
    suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'gq']
    features['suspicious_tld'] = 1 if ext.suffix in suspicious_tlds else 0
    
    # HTTPS
    features['https'] = 1 if parsed.scheme == 'https' else 0
    
    return features

def build_feature_dataframe(df, url_column):
    feature_list = df[url_column].apply(lambda x: extract_features(x))
    feature_df = pd.DataFrame(feature_list.tolist())
    return feature_df

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv('Train2.csv')

In [7]:
df.head(5)

,url,Phish?
0,http://elretohistorico.com/tag/bruja,0
1,http://fcouncil.ncku.edu.tw/ebsmeeting_1.aspx,0
2,http://q-r.to/bg6qsB,1
3,http://www.polaroid-passion.com/forum/viewforu...,0
4,http://docs.google.com/presentation/d/e/2PACX-...,1


In [8]:
data = build_feature_dataframe(df, 'url')

In [9]:
data['Target'] = df['Phish?']

In [10]:
data.to_csv('Moksh_testing.csv')

In [11]:
data

,url_length,domain_length,subdomain_length,path_length,._count,-_count,__count,/_count,?_count,=_count,...,special_char_count,digit_ratio,letter_ratio,has_ip,entropy,word_count,keyword_count,suspicious_tld,https,Target
0,36,15,0,10,1,0,0,4,0,0,...,6,0.000000,0.833333,0,3.993133,5,0,0,0,0
1,45,4,8,18,4,0,1,3,0,0,...,9,0.022222,0.777778,0,4.402530,8,0,0,0,0
2,20,3,0,7,1,1,0,3,0,0,...,6,0.050000,0.650000,0,3.746439,5,0,0,0,1
3,60,16,3,20,3,1,0,4,1,1,...,12,0.033333,0.766667,0,4.460324,11,0,0,0,0
4,174,6,4,108,2,1,0,7,1,3,...,19,0.109195,0.781609,0,5.574198,18,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111396,147,11,0,125,1,16,0,6,0,0,...,24,0.054422,0.782313,0,4.445113,23,0,0,0,0
111397,32,7,3,8,3,0,0,3,0,0,...,7,0.000000,0.781250,0,3.905639,6,0,0,0,0
111398,50,6,3,29,2,1,0,5,0,0,...,9,0.040000,0.780000,0,4.351272,8,0,0,0,0
111399,20,6,0,3,1,0,0,3,0,0,...,5,0.000000,0.750000,0,3.608695,4,0,0,0,1


In [21]:
df1 = pd.read_csv('Train3.csv')

In [22]:
df1

,url,Phish
0,rjop.com/osier.html,1
1,www.kingshealthpartners.org/latest/317-joining...,1
2,www.1588.lt/en/company/urticae-vaistine-uab-fi...,1
3,79.docs.google.com/,1
4,calpa.info/?q=calendar-node-field-date/day/201...,1
...,...,...
49995,www.bjycgm.com/xinwen/gongsixinwen/601.html,0
49996,ecuadorvscolombialive.us/23e4aduse99dd_q028dd_...,0
49997,joachimpimmelbergerswelt.wordpress.com/woche2-8/,0
49998,www.cartesfrance.fr/carte-france-ville/24022_b...,0


In [23]:
data = build_feature_dataframe(df1, 'url')
data['Target'] = df1['Phish']
data['Target'].fillna(0.0)

0        1
1        1
2        1
3        1
4        1
        ..
49995    0
49996    0
49997    0
49998    0
49999    0
Name: Target, Length: 50000, dtype: int64

In [24]:
data.to_csv('Aryan_testing.csv')

In [25]:
data

,url_length,domain_length,subdomain_length,path_length,._count,-_count,__count,/_count,?_count,=_count,...,special_char_count,digit_ratio,letter_ratio,has_ip,entropy,word_count,keyword_count,suspicious_tld,https,Target
0,19,4,0,19,2,0,0,1,0,0,...,3,0.000000,0.842105,0,3.681881,4,0,0,0,1
1,87,19,3,87,2,8,0,2,0,0,...,12,0.034483,0.827586,0,4.357885,13,0,0,0,1
2,77,4,3,77,2,6,1,3,0,0,...,12,0.064935,0.779221,0,4.326771,13,0,0,0,1
3,19,6,7,19,3,0,0,1,0,0,...,4,0.105263,0.684211,0,3.366091,4,0,0,0,1
4,53,5,0,11,1,5,0,3,1,1,...,11,0.150943,0.641509,0,4.298701,11,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,43,6,3,43,3,0,0,3,0,0,...,6,0.069767,0.790698,0,4.147341,7,0,0,0,0
49996,126,21,0,126,2,0,6,1,0,0,...,12,0.349206,0.555556,0,4.626738,13,0,0,0,0
49997,48,9,24,48,2,1,0,2,0,0,...,5,0.041667,0.854167,0,4.214662,5,0,0,0,0
49998,71,12,3,71,3,4,1,2,0,0,...,10,0.070423,0.788732,0,4.418033,11,0,0,0,0
